# CLAP Embedding Visualisation

Visualise the 512-dimensional CLAP audio embeddings using t-SNE and UMAP projections, coloured by genre.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

from src.config import PROCESSED_DIR
from src.metadata import load_tracks

sns.set_style('whitegrid')
%matplotlib inline

## Load Embeddings & Metadata

In [ ]:
embeddings = np.load(PROCESSED_DIR / 'clap_embeddings.npy')
track_ids = np.load(PROCESSED_DIR / 'clap_track_ids.npy')
tracks = load_tracks()

# Get genre labels for each embedded track
genres = [tracks.loc[tid].get(('track', 'genre_top'), 'Unknown') for tid in track_ids]
genres = pd.Series(genres, name='genre')

print(f'Embeddings: {embeddings.shape}')
print(f'Genre distribution:\n{genres.value_counts()}')

## PCA — Variance Explained

In [ ]:
pca_full = PCA(n_components=50).fit(embeddings)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 51), cumvar, 'o-', markersize=4)
ax.axhline(y=0.9, color='r', linestyle='--', alpha=0.5, label='90% variance')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('PCA — Cumulative Variance Explained (CLAP Embeddings)')
ax.legend()
plt.tight_layout()
plt.show()

n90 = np.argmax(cumvar >= 0.9) + 1
print(f'Components for 90% variance: {n90}')

## t-SNE Projection (coloured by genre)

In [ ]:
# PCA to 50 dims first for speed
pca50 = PCA(n_components=50).fit_transform(embeddings)

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
tsne_2d = tsne.fit_transform(pca50)

df_tsne = pd.DataFrame({'x': tsne_2d[:, 0], 'y': tsne_2d[:, 1], 'genre': genres})
print('t-SNE done.')

In [ ]:
top_genres = genres.value_counts().head(8).index.tolist()
df_plot = df_tsne[df_tsne['genre'].isin(top_genres)].copy()

fig, ax = plt.subplots(figsize=(12, 10))
palette = sns.color_palette('husl', len(top_genres))
for i, genre in enumerate(top_genres):
    mask = df_plot['genre'] == genre
    ax.scatter(df_plot.loc[mask, 'x'], df_plot.loc[mask, 'y'],
               s=8, alpha=0.6, label=genre, color=palette[i])
ax.legend(markerscale=3, fontsize=10)
ax.set_title('t-SNE of CLAP Audio Embeddings (top 8 genres)', fontsize=14)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig('../data/processed/tsne_clap_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

## PCA 2D Projection

In [ ]:
pca2 = PCA(n_components=2).fit_transform(embeddings)
df_pca = pd.DataFrame({'x': pca2[:, 0], 'y': pca2[:, 1], 'genre': genres})
df_pca_plot = df_pca[df_pca['genre'].isin(top_genres)].copy()

fig, ax = plt.subplots(figsize=(12, 10))
for i, genre in enumerate(top_genres):
    mask = df_pca_plot['genre'] == genre
    ax.scatter(df_pca_plot.loc[mask, 'x'], df_pca_plot.loc[mask, 'y'],
               s=8, alpha=0.6, label=genre, color=palette[i])
ax.legend(markerscale=3, fontsize=10)
ax.set_title('PCA of CLAP Audio Embeddings (top 8 genres)', fontsize=14)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
plt.tight_layout()
plt.savefig('../data/processed/pca_clap_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

## Cosine Similarity Heatmap (genre centroids)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute genre centroids
centroids = []
for genre in top_genres:
    mask = (genres == genre).values
    centroids.append(embeddings[mask].mean(axis=0))
centroids = np.array(centroids)

sim_matrix = cosine_similarity(centroids)

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(sim_matrix, xticklabels=top_genres, yticklabels=top_genres,
            annot=True, fmt='.2f', cmap='RdYlBu_r', vmin=0.5, vmax=1.0, ax=ax)
ax.set_title('Cosine Similarity Between Genre Centroids (CLAP)', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/genre_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Embedding Norm Distribution

In [ ]:
norms = np.linalg.norm(embeddings, axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(norms, bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('L2 Norm')
ax.set_ylabel('Count')
ax.set_title('Distribution of CLAP Embedding Norms')
plt.tight_layout()
plt.show()
print(f'Norm — mean: {norms.mean():.3f}, std: {norms.std():.3f}, min: {norms.min():.3f}, max: {norms.max():.3f}')